# 08B — Original mapped firelines, scale, and selection sensitivity

**Research question:** Do the condition–outcome patterns depend on dividing firelines into ≤250 m fireline units, selecting a subset of the archive, or weighting fires and records differently?

Run all cells in Antigravity with the project's **.venv** kernel. This is a self-contained exploratory analysis. It reads existing files, does not download data or fit models, and writes only to a new timestamped output folder.

## The levels we can actually study
- **Version-resolved source record:** one mapped NIFC geometry after duplicate database versions were resolved. Source geometries already have outcome labels and may contain several stored parts. A record is **not necessarily a complete constructed fireline**.
- **Represented source record:** a source record with at least one fireline unit in the final 304,143-fireline-unit dataset.
- **Fireline unit:** the analytical ≤250 m piece already used for the prediction study.
- **Fire:** all records sharing the source fire identifier.

The archive has no verified construction-asset identifier. We will not join adjacent records into an invented whole fireline, connect gaps, combine different outcomes, or interpret database mapping times as construction times.

## What this notebook does
1. Analyses the original source archive, including **Not Engaged as a separate outcome**.
2. Independently calculates geodesic geometry lengths and audits multipart geometry, width, reported length, and dates.
3. Measures which source records were represented after our historical selection.
4. Compares equal-fireline-unit, equal-source-record, and mapped-length weighting on matched data.
5. Summarizes retained fireline units within each source record, retaining timing spread, missingness, and length-ratio diagnostics.
6. Tests whether selected condition associations survive weighting changes and repeats them across years and fireline types.
7. Checks canopy availability explicitly, including predictors sampled directly on the fireline.

## Boundaries
The environmental summaries describe **retained portions**, not unsampled parts of original records. They are not new forecasts at a single original-fireline prediction time. Source geometry lengths may depend on how outcome portions were mapped, so length associations cannot establish an optimal construction length.

This is broader than 08A but not an exhaustive search for favorable results. Fixed bins and comparisons are recorded before the run; none are chosen automatically for significance. Whole-fire intervals do not correct selection bias or multiple exploratory comparisons.

**Next stages:** condition-rule development and excluded-fire/year evaluation belong in **08C** after this audit. Genuine 100 m/500 m reconstruction, independent NIFC–NIROPS agreement, construction-data acquisition, and prospective evaluation remain separate research tracks.


## 0. Configuration and fixed inputs

The completed 08A run below is pinned explicitly. Source-record results do not depend on inheriting repeated widths from its fireline units: original attributes are loaded directly from the source archive.

The geometry pass streams the source file in batches. It can take several minutes and prints progress. It does not require a GPU. Bootstrap counts and review thresholds are configurable, but changing them creates another exploratory run.


In [ ]:
import os  # Configurable project root for the public release.
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, sys, platform, uuid, zlib, time, importlib.util
from importlib.metadata import version as package_version

required = ["numpy", "pandas", "pyarrow", "shapely", "pyproj", "matplotlib", "IPython"]
missing = [x for x in required if importlib.util.find_spec(x) is None]
if missing:
    raise ImportError("Use the project .venv kernel. Missing packages: " + ", ".join(missing))
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import shapely
from shapely.geometry import LineString, MultiLineString
from pyproj import CRS, Geod
import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
from IPython.display import display, Markdown

def find_root():
    for p in [Path.cwd(), *Path.cwd().parents, Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))]:
        if (p / "data/processed/fireline/fireline_qaqc_version_resolved.parquet").is_file():
            return p.resolve()
    raise FileNotFoundError("Open the notebook inside the project or set PROJECT_ROOT.")
PROJECT_ROOT = find_root()
INPUT_08A_RUN_NAME = "08A_20260930T220658_104767Z_614ae3"
INPUT_08A = PROJECT_ROOT / "outputs/08_condition_outcomes" / INPUT_08A_RUN_NAME
SOURCE_PATH = PROJECT_ROOT / "data/processed/fireline/fireline_qaqc_version_resolved.parquet"
MASTER_PATH = PROJECT_ROOT / "data/processed/modeling_table_assembly_v1/fireline_hold_modeling_master_v1.parquet"
UNIT_PATH = INPUT_08A / "08A_enriched_frozen_fireline_units.parquet"
SOURCE_YEARS = list(range(2017, 2025))
COMPARISON_YEARS = list(range(2020, 2025))
RANDOM_SEED = 20260930
BOOTSTRAP_REPLICATES = 2000
GEOMETRY_BATCH_SIZE = 4096
WIDTH_REVIEW_FT = 300.0  # Review trigger, not a validated maximum feasible width.
MIN_GROUP_ROWS = 200
MIN_GROUP_FIRES = 20
MIN_GROUP_OUTCOME_COUNT = 20
MIN_CONTRAST_UNITS_PER_SIDE = 5
MIN_CONTRAST_PARENTS_PER_SIDE = 2
MIN_ORIGINAL_RECORDS_PER_SIDE = 5
HIGH_RETAINED_LENGTH_RATIO = 0.80  # Diagnostic subset, not verified spatial coverage.
RETENTION_RATIO_REVIEW_LIMIT = 1.02
IS_SMOKE_TEST = False
BUILD_ID = "08B_V1_ORIGINAL_RECORD_SCALE_AND_SELECTION"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ") + "_" + uuid.uuid4().hex[:6]
RUN_DIR = PROJECT_ROOT / "outputs/08_condition_outcomes" / ("08B_" + RUN_ID)
FIG_DIR = RUN_DIR / "figures"
NOTEBOOK_PATH = PROJECT_ROOT / "notebooks/08B_original_firelines_scale_and_selection.ipynb"

# Each predictor retains its specific physical meaning; no new prediction time is assigned.
PREDICTORS = {
 "humidity": ("hrrr_window_0_18h_rh2_pct_mean", "Mean forecast relative humidity, 0–18 h", "%", [0,20,40,60,80,np.inf], ["0–<20","20–<40","40–<60","60–<80","80–100"], (0,100)),
 "gust": ("hrrr_gust_mps_p18h", "Forecast gust at +18 h", "m/s", [0,5,10,15,20,np.inf], ["0–<5","5–<10","10–<15","15–<20","20+"], (0,None)),
 "wind": ("hrrr_window_0_18h_wind10_speed_mean_mps", "Mean forecast wind, 0–18 h", "m/s", [0,3,6,9,12,np.inf], ["0–<3","3–<6","6–<9","9–<12","12+"], (0,None)),
 "temperature": ("hrrr_window_0_18h_t2_c_max", "Maximum forecast temperature, 0–18 h", "C", [-np.inf,10,20,30,40,np.inf], ["<10","10–<20","20–<30","30–<40","40+"], (None,None)),
 "slope": ("terrain_slope_mean_deg", "Mean terrain slope", "degrees", [0,10,20,30,40,np.inf], ["0–<10","10–<20","20–<30","30–40","40+"], (0,90)),
 "elevation": ("terrain_elevation_mean_m", "Mean elevation", "m", [-np.inf,500,1000,1500,2000,np.inf], ["<500","500–<1000","1000–<1500","1500–<2000","2000+"], (None,None)),
 "opposite_crown": ("landfire_protected_far_crown_depth_p90", "Opposite-side far crown depth p90", "m", [0,5,10,20,30,np.inf], ["0–<5","5–<10","10–<20","20–<30","30+"], (0,None)),
 "line_crown": ("landfire_line_crown_depth_mean", "Crown depth sampled on the fireline, mean", "m", [0,5,10,20,30,np.inf], ["0–<5","5–<10","10–<20","20–<30","30+"], (0,None)),
 "line_cover": ("landfire_line_cc_mean", "Canopy cover sampled on the fireline, mean", "%", [0,20,40,60,80,np.inf], ["0–<20","20–<40","40–<60","60–<80","80–100"], (0,100)),
 "distance": ("local_fire_to_line_distance_geom_m", "Mapped fire-perimeter distance", "m", [0,25,50,100,250,1000,np.inf], ["0–<25","25–<50","50–<100","100–<250","250–<1000","1000+"], (0,None)),
}
WIDTH_EDGES = [0,3,6,12,30,np.inf]
WIDTH_LABELS = [">0–<3","3–<6","6–<12","12–<30","30+"]
LENGTH_EDGES = [0,50,100,250,500,1000,np.inf]
LENGTH_LABELS = [">0–<50","50–<100","100–<250","250–<500","500–<1000","1000+"]
OUTCOMES = ["Held", "Burned Over", "Not Engaged"]
pd.set_option("display.max_rows", 25)
pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.dpi":110, "savefig.dpi":220,
                     "axes.spines.top":False, "axes.spines.right":False})
print(BUILD_ID, "\n08A input:", INPUT_08A, "\nNew output:", RUN_DIR)


## 1. Calculation helpers

Binary Held fractions below use only **Held and Burned Over** records. Three-outcome tables have their own explicit denominator.

A pooled fraction weights all observations according to the stated scheme. A fire-average fraction first calculates the weighted fraction within each fire, then gives each fire equal weight. Difference intervals use the **same fires** on both sides. Sparse groups remain visible and are flagged.

Retained-length ratios use a sum of child lengths, not a spatial union. They can reveal incomplete representation or inconsistencies but do not prove geometric coverage.


In [ ]:
def require(ok, message):
    if not bool(ok): raise AssertionError(message)

def unique(frame, keys, name):
    require(frame[keys].notna().all().all(), name + ": missing key")
    require(not frame.duplicated(keys).any(), name + ": duplicate keys")

def read_cols(path, columns):
    require(path.is_file(), "Missing input: " + str(path))
    require(set(columns).issubset(pq.ParquetFile(path).schema_arrow.names), "Missing columns: " + str(path))
    return pd.read_parquet(path, columns=list(dict.fromkeys(columns)))

def digest(frame, columns, keys):
    x = frame[columns].sort_values(keys).reset_index(drop=True)
    return hashlib.sha256(pd.util.hash_pandas_object(x,index=False).to_numpy().tobytes()).hexdigest()

def safe_json(value):
    if isinstance(value, Path): return str(value)
    if isinstance(value, dict): return {str(k):safe_json(v) for k,v in value.items()}
    if isinstance(value, (list,tuple)): return [safe_json(v) for v in value]
    if isinstance(value, (np.integer,)): return int(value)
    if isinstance(value, (np.bool_,)): return bool(value)
    if isinstance(value, (np.floating,float)):
        if np.isnan(value): return None
        if np.isposinf(value): return "+inf"
        if np.isneginf(value): return "-inf"
        return float(value)
    return value

def write_json(name,value):
    (RUN_DIR/name).write_text(json.dumps(safe_json(value),indent=2,allow_nan=False),encoding="utf-8")

def fire_interval(values, key):
    x=np.asarray(values,dtype=float)
    require(np.isfinite(x).all(), "Nonfinite fire score")
    if len(x)<2: return (np.nan,np.nan)
    rng=np.random.default_rng((RANDOM_SEED+zlib.crc32(str(key).encode()))%(2**32))
    means=np.empty(BOOTSTRAP_REPLICATES)
    for start in range(0,BOOTSTRAP_REPLICATES,128):
        stop=min(start+128,BOOTSTRAP_REPLICATES)
        means[start:stop]=x[rng.integers(0,len(x),(stop-start,len(x)))].mean(axis=1)
    return tuple(np.quantile(means,[0.025,0.975]))

def numeric_bins(series,edges,labels,bounds=(None,None)):
    x=pd.to_numeric(series,errors="coerce")
    valid=pd.Series(np.isfinite(x),index=x.index)
    lo,hi=bounds
    if lo is not None: valid &= x.ge(lo)
    if hi is not None: valid &= x.le(hi)
    out=pd.Series("Missing",index=x.index,dtype="string")
    out.loc[series.notna() & x.isna()]="Non-numeric"
    out.loc[x.notna() & ~valid]="Invalid / nonfinite"
    out.loc[valid]=pd.cut(x[valid],edges,labels=labels,right=False,include_lowest=True).astype("string")
    require(out.notna().all(),"Bin edges do not cover valid observations")
    return out

def width_status(values):
    x=pd.to_numeric(values,errors="coerce")
    finite=pd.Series(np.isfinite(x),index=x.index)
    s=pd.Series("Missing",index=x.index,dtype="string")
    s.loc[values.notna() & x.isna()]="Non-numeric"
    s.loc[x.notna() & ~finite]="Nonfinite"
    s.loc[finite & x.lt(0)]="Negative"
    s.loc[finite & x.eq(0)]="Zero"
    s.loc[finite & x.gt(0) & x.le(WIDTH_REVIEW_FT)]="Positive, at or below review flag"
    s.loc[finite & x.gt(WIDTH_REVIEW_FT)]="Positive, above review flag"
    return s

def describe_numbers(values):
    x=pd.to_numeric(values,errors="coerce")
    v=x[np.isfinite(x)]
    q=v.quantile([.05,.25,.5,.75,.95])
    return {"rows":len(x),"missing":int(x.isna().sum()),
            "nonfinite_nonmissing":int((x.notna() & ~np.isfinite(x)).sum()),
            "zero":int(x.eq(0).sum()),"negative":int(x.lt(0).sum()),
            "mean":v.mean(),"median":v.median(),"min":v.min(),"p05":q.loc[.05],
            "p25":q.loc[.25],"p75":q.loc[.75],"p95":q.loc[.95],"max":v.max()}

def fire_scores(frame,weight=None):
    require(frame["y"].isin([0,1]).all(),"Binary summaries must exclude Not Engaged and missing outcomes.")
    w=pd.Series(1.,index=frame.index) if weight is None else pd.to_numeric(frame[weight],errors="coerce")
    valid=np.isfinite(w) & w.gt(0)
    x=frame.loc[valid,["_irwin_norm","y"]].copy()
    x["denominator"]=w.loc[valid]
    x["numerator"]=x["y"]*x["denominator"]
    g=x.groupby("_irwin_norm")[["numerator","denominator"]].sum()
    g["score"]=g["numerator"]/g["denominator"]
    return g,valid

def binary_summary(frame,groups,analysis,weight=None,intervals=True):
    rows=[]
    if not groups:
        iterator=[((),frame)]
    else:
        iterator=frame.groupby(groups[0] if len(groups)==1 else groups,dropna=False,observed=True,sort=True)
    for keys,part in iterator:
        keys=(keys,) if len(groups)==1 else keys
        scores,valid=fire_scores(part,weight)
        used=part.loc[valid]
        n=len(used); h=int(used["y"].sum())
        rates=scores["score"].to_numpy()
        # Matching fire populations use identical resamples across weighting schemes.
        ci=fire_interval(rates,("binary fire mean",tuple(scores.index))) if intervals and len(rates) else (np.nan,np.nan)
        row=dict(zip(groups,keys))
        row.update(analysis=analysis,weighting=weight or "equal_observation",
                   n_observations=len(part),n_used=n,n_invalid_weights=int((~valid).sum()),
                   n_source_records=used["source_record_key"].nunique(),n_fires=len(scores),
                   n_held=h,n_burned_over=n-h,
                   pooled_held_rate=scores["numerator"].sum()/scores["denominator"].sum() if len(scores) else np.nan,
                   fire_average_held_rate=rates.mean() if len(rates) else np.nan,
                   held_ci_lower=ci[0],held_ci_upper=ci[1],
                   meets_reporting_screen=bool(n>=MIN_GROUP_ROWS and len(scores)>=MIN_GROUP_FIRES and min(h,n-h)>=MIN_GROUP_OUTCOME_COUNT),
                   bootstrap_degenerate=bool(len(rates)>1 and np.ptp(rates)==0))
        rows.append(row)
    return pd.DataFrame(rows)

def paired_score_difference(a,b,name):
    pair=pd.concat([a.rename("A"),b.rename("B")],axis=1).dropna().sort_index()
    d=pair["B"]-pair["A"]
    lo,hi=fire_interval(d,("paired fire difference",tuple(pair.index)))
    row={"comparison":name,"n_common_fires":len(pair),"mean_A":pair["A"].mean(),
         "mean_B":pair["B"].mean(),"difference_B_minus_A":d.mean(),
         "ci_lower":lo,"ci_upper":hi,
         "evidence":"Exploratory; same fires; not causal"}
    detail=pair.assign(difference_B_minus_A=d,comparison=name).reset_index()
    return row,detail

RUN_DIR.mkdir(parents=True,exist_ok=False)
FIG_DIR.mkdir()
tables={}
figures=[]
print("Helpers ready. No source files will be written.")


## 2. Load original records and verify the 08A population

The full source archive covers 2017–2024. Environmental comparisons use the existing 2020–2024 prediction population. Older years are reported separately, not supplied with invented weather or prediction times.

The source archive has already resolved repeated database versions. We do not count all raw database-history rows as independent firelines.


In [ ]:
manifest_a=json.loads((INPUT_08A/"08A_run_manifest.json").read_text(encoding="utf-8"))
checks_a=json.loads((INPUT_08A/"08A_quality_checks.json").read_text(encoding="utf-8"))
require(not manifest_a["is_smoke_test"],"08B requires a completed full 08A run, not a smoke test.")
require(checks_a["master_population_preserved"],"08A population check failed.")
source_columns=["_irwin_norm","_geom_hash","Year","_engagement_norm","FeatureCategory",
                "StrategicLineType","MapMethod","LineWidthFeet","LengthFeet",
                "LineLengthGeodesicKM","_length_km_geom","LineDateTime","CreateDate","DateCurrent"]
source=read_cols(SOURCE_PATH,source_columns)
unique(source,["_irwin_norm","_geom_hash"],"Original version-resolved mapped records")
require(len(source)==570111 and source["_irwin_norm"].nunique()==1158,"Unexpected source archive version.")
require(set(source["_engagement_norm"])==set(OUTCOMES),"Unrecognized source labels.")
require(source.groupby("_irwin_norm")["Year"].nunique().eq(1).all(),"Source fire identifiers span multiple years; review grouping.")
source["source_record_key"]=source["_irwin_norm"].astype(str)+"|"+source["_geom_hash"].astype(str)
unique(source,["source_record_key"],"Source keys")
source["fireline_type"]=source["FeatureCategory"].astype("string").fillna("Missing")
source["y"]=source["_engagement_norm"].map({"Held":1,"Burned Over":0})
source["width_status"]=width_status(source["LineWidthFeet"])
source["reported_width_m"]=pd.to_numeric(source["LineWidthFeet"],errors="coerce")*.3048
source["screened_width_m"]=source["reported_width_m"].where(source["width_status"].eq("Positive, at or below review flag"))
source["bin_width"]=numeric_bins(source["screened_width_m"],WIDTH_EDGES,WIDTH_LABELS,(0,None))
source.loc[source["screened_width_m"].isna(),"bin_width"]="Width: "+source["width_status"]
source["source_length_5070_m"]=source["_length_km_geom"]*1000
source["reported_length_m"]=source["LengthFeet"]*.3048
source["stored_geodesic_length_m"]=source["LineLengthGeodesicKM"]*1000
source_start_hash=digest(source,source_columns,["_irwin_norm","_geom_hash"])

unit_columns=["unit_uid","_irwin_norm","_geom_hash","source_record_key","Year","hold_label",
              "prediction_anchor_utc","exact_t0_observation_utc","actual_line_length_m",
              "outer_fold","FeatureCategory","LineWidthFeet"]
unit_columns += [spec[0] for key,spec in PREDICTORS.items() if key not in ["line_crown","line_cover"]]
units=read_cols(UNIT_PATH,unit_columns)
unique(units,["unit_uid"],"08A fireline units")
require(len(units)==304143 and units["_irwin_norm"].nunique()==381,"Unexpected final population.")
identity=["unit_uid","_irwin_norm","Year","hold_label","prediction_anchor_utc","exact_t0_observation_utc"]
extra=["landfire_line_crown_depth_mean","landfire_line_cc_mean",
       "landfire_line_derived_canopy_joint_valid_weight_fraction",
       "approach_direction_reliable","landfire_directional_context_valid"]
master_columns=identity+[spec[0] for spec in PREDICTORS.values()]+extra
master=read_cols(MASTER_PATH,master_columns)
unique(master,["unit_uid"],"Master")
require(digest(units,identity,["unit_uid"])==digest(master,identity,["unit_uid"]),"08A/master identities or anchors disagree.")
compare_columns=[x for x in units.columns if x in master.columns]
require(digest(units,compare_columns,["unit_uid"])==digest(master,compare_columns,["unit_uid"]),"08A selected predictors differ from master.")
units=units.merge(master[["unit_uid"]+extra],on="unit_uid",how="left",validate="one_to_one")
units["y"]=units["hold_label"]
units["fireline_type"]=units["FeatureCategory"]
units["observation_weight"]=1.0
unit_base_columns=units.columns.tolist()
units_start_hash=digest(units,unit_base_columns,["unit_uid"])
require(units["actual_line_length_m"].between(0,250.000001,inclusive="right").all(),"Invalid analytical length.")
represented=set(units["source_record_key"])
require(len(represented)==142999,"Unexpected represented source-record count.")
source["represented"]=source["source_record_key"].isin(represented)
source["in_study_years"]=source["Year"].isin(COMPARISON_YEARS)
source["in_study_fires"]=source["_irwin_norm"].isin(units["_irwin_norm"])
unit_match=units.merge(source[["source_record_key","_irwin_norm","Year","y","FeatureCategory","LineWidthFeet"]],
                       on="source_record_key",how="left",validate="many_to_one",suffixes=("","_source"),indicator=True)
require(unit_match["_merge"].eq("both").all(),"A fireline unit has no source record.")
for name in ["_irwin_norm","Year","y","FeatureCategory","LineWidthFeet"]:
    same=unit_match[name].eq(unit_match[name+"_source"]) | (unit_match[name].isna() & unit_match[name+"_source"].isna())
    require(same.all(),"Source metadata disagreement: "+name)
input_paths=[SOURCE_PATH,MASTER_PATH,UNIT_PATH,INPUT_08A/"08A_run_manifest.json",INPUT_08A/"08A_quality_checks.json"]
input_stats={str(p):{"bytes":p.stat().st_size,"modified_ns":p.stat().st_mtime_ns} for p in input_paths}
print(f"Original archive: {len(source):,} records / {source['_irwin_norm'].nunique():,} fires.")
print(f"Final population: {len(units):,} fireline units / {len(represented):,} source records.")


## 3. Independent geometry audit

Lengths are recomputed on the WGS84 ellipsoid from the archived geometry. Each stored part is measured without filling its gaps. The stored source-record length field used EPSG:5070; corrected analytical fireline-unit lengths used local projections. Differences are audited rather than silently overwritten.

A multipart record is not automatically several disconnected constructed firelines. The audit counts **stored geometry parts** only. Coarse geographic boxes help detect projection issues; they are not verified state boundaries.

No invalid geometries are repaired here. Problems remain flagged, and weighted analyses report observations without a usable length.


In [ ]:
geo_meta=json.loads(pq.ParquetFile(SOURCE_PATH).schema_arrow.metadata[b"geo"])
geo_column=geo_meta["primary_column"]
require(CRS.from_user_input(geo_meta["columns"][geo_column]["crs"]).equals(CRS.from_epsg(4326)),
        "Expected WGS84 geometry. Review CRS before computing lengths.")
geod=Geod(ellps="WGS84")
geometry_rows=[]
started=time.perf_counter()
for batch_no,batch in enumerate(pq.ParquetFile(SOURCE_PATH).iter_batches(
        batch_size=GEOMETRY_BATCH_SIZE,columns=["_irwin_norm","_geom_hash",geo_column])):
    data=batch.to_pydict()
    for fire,key,wkb in zip(data["_irwin_norm"],data["_geom_hash"],data[geo_column]):
        row={"source_record_key":str(fire)+"|"+str(key),"geodesic_length_m":np.nan,
             "geometry_parts":np.nan,"geometry_vertices":np.nan,"geometry_simple":False,
             "geographic_box":"Unavailable","geometry_issue":"ok"}
        try:
            shape=shapely.from_wkb(wkb)
            if shape is None or shape.is_empty:
                row["geometry_issue"]="missing_or_empty"
            elif shape.geom_type not in ["LineString","MultiLineString"]:
                row["geometry_issue"]="unexpected_geometry_type"
            else:
                bounds=np.asarray(shape.bounds)
                coords=shapely.get_coordinates(shape)
                valid_coordinates=(np.isfinite(coords).all() and
                                   np.abs(coords[:,0]).max()<=180 and np.abs(coords[:,1]).max()<=90)
                row["geometry_parts"]=len(shape.geoms) if shape.geom_type=="MultiLineString" else 1
                row["geometry_vertices"]=len(coords)
                row["geometry_simple"]=bool(shape.is_simple)
                if not valid_coordinates or not shape.is_valid:
                    row["geometry_issue"]="invalid_geometry_or_coordinates"
                else:
                    length=float(geod.geometry_length(shape))
                    row["geodesic_length_m"]=length if np.isfinite(length) and length>0 else np.nan
                    if not np.isfinite(row["geodesic_length_m"]):
                        row["geometry_issue"]="nonpositive_or_nonfinite_length"
                    lon=(bounds[0]+bounds[2])/2
                    lat=(bounds[1]+bounds[3])/2
                    row["geographic_box"]=(
                        "Lower-48 box" if -125<=lon<=-66 and 24<=lat<=50 else
                        "Alaska box" if -180<=lon<=-129 and 50<lat<=72 else
                        "Hawaii box" if -161<=lon<=-154 and 18<=lat<=23 else "Other")
        except Exception as exc:
            row["geometry_issue"]="decode_or_length_error:"+type(exc).__name__
        geometry_rows.append(row)
    if batch_no%20==0:
        print(f"Geometry audit: {len(geometry_rows):,}/{len(source):,} records, "
              f"{time.perf_counter()-started:.1f} seconds",flush=True)
geometry=pd.DataFrame(geometry_rows)
unique(geometry,["source_record_key"],"Geometry audit")
require(len(geometry)==len(source),"Geometry pass lost source records.")
source=source.merge(geometry,on="source_record_key",how="left",validate="one_to_one")
source["geometry_part_group"]=pd.cut(source["geometry_parts"],[0,1,5,20,np.inf],
                                    labels=["1","2–5","6–20","21+"],right=True).astype("string").fillna("Unavailable")
source["bin_length"]=numeric_bins(source["geodesic_length_m"],LENGTH_EDGES,LENGTH_LABELS,(0,None))
source["ratio_5070_to_geodesic"]=source["source_length_5070_m"]/source["geodesic_length_m"]
source["ratio_reported_to_geodesic"]=source["reported_length_m"].where(source["reported_length_m"].gt(0))/source["geodesic_length_m"]
source["ratio_stored_geodesic_to_recomputed"]=source["stored_geodesic_length_m"].where(source["stored_geodesic_length_m"].gt(0))/source["geodesic_length_m"]
tables["geometry_issue_counts"]=source.groupby(["geographic_box","geometry_issue"],dropna=False).size().rename("n_source_records").reset_index()
tables["geometry_length_diagnostics"]=pd.DataFrame([
    {"geographic_box":box,"quantity":col,**describe_numbers(part[col])}
    for box,part in source.groupby("geographic_box")
    for col in ["geometry_parts","geometry_vertices","geodesic_length_m",
                "ratio_5070_to_geodesic","ratio_reported_to_geodesic","ratio_stored_geodesic_to_recomputed"]])
print("Geometry audit complete. Non-simple geometry is a diagnostic, not proof of invalidity.")
display(tables["geometry_issue_counts"])


## 4. Populations, original outcomes, and selection

The population table deliberately mixes neither row counts nor meanings. Source-record inclusion percentages use source-record denominators; the fireline-unit count is a separate level.

Original records with **Not Engaged** labels stay in the three-outcome tables. Held-versus-Burned Over rates are explicitly conditional on those two outcomes. Comparing selected and unselected records is a selection diagnostic, not an estimate of a causal selection effect.

The same-fire comparison below uses only fires containing both represented and unrepresented engaged source records.


In [ ]:
population_masks={
 "Source archive 2017–2024":pd.Series(True,index=source.index),
 "Source records 2020–2024":source["in_study_years"],
 "Source records in the 381 study fires":source["in_study_years"] & source["in_study_fires"],
 "Represented source records":source["represented"],
}
population_rows=[]; class_rows=[]; original_profiles=[]
for name,mask in population_masks.items():
    part=source.loc[mask]
    engaged=part.loc[part["y"].notna()]
    population_rows.append({"population":name,"observation_level":"source record","n_observations":len(part),
                           "n_fires":part["_irwin_norm"].nunique(),"n_held":int(part["y"].eq(1).sum()),
                           "n_burned_over":int(part["y"].eq(0).sum()),
                           "n_not_engaged":int(part["_engagement_norm"].eq("Not Engaged").sum()),
                           "n_usable_geometry_lengths":int(part["geodesic_length_m"].notna().sum())})
    for outcome in OUTCOMES:
        sub=part.loc[part["_engagement_norm"].eq(outcome)]
        length=0.0 if len(sub)==0 else sub["geodesic_length_m"].sum(min_count=1)
        total=part["geodesic_length_m"].sum(min_count=1)
        class_rows.append({"population":name,"outcome":outcome,
                           "n_source_records":len(sub),"record_fraction":len(sub)/len(part),
                           "mapped_km":length/1000,"mapped_length_fraction":length/total,
                           "n_fires":sub["_irwin_norm"].nunique()})
    original_profiles.append(binary_summary(engaged,[],name+": Held versus Burned Over"))
population_rows.append({"population":"Final fireline units","observation_level":"fireline unit",
                        "n_observations":len(units),"n_fires":units["_irwin_norm"].nunique(),
                        "n_held":int(units["y"].eq(1).sum()),"n_burned_over":int(units["y"].eq(0).sum()),
                        "n_not_engaged":0,"n_usable_geometry_lengths":np.nan})
tables["population_inventory"]=pd.DataFrame(population_rows)
tables["source_three_outcome_composition"]=pd.DataFrame(class_rows)
tables["source_binary_outcomes_by_population"]=pd.concat(original_profiles,ignore_index=True)
tables["source_outcome_counts_by_year"]=source.groupby(["Year","_engagement_norm"]).size().rename("n_source_records").reset_index()

shared=source.loc[source["in_study_years"] & source["in_study_fires"]].copy()
selection_rows=[]
for dimension in ["_engagement_norm","Year","fireline_type","width_status","bin_length","geometry_part_group"]:
    for value,part in shared.groupby(dimension,dropna=False,observed=True):
        selection_rows.append({"dimension":dimension,"group":value,"n_source_records":len(part),
                               "n_represented":int(part["represented"].sum()),
                               "represented_fraction":part["represented"].mean(),
                               "n_fires":part["_irwin_norm"].nunique()})
tables["source_record_selection_by_attribute"]=pd.DataFrame(selection_rows)
original_attribute_profiles=[]
for name,frame in [("All 2020–2024 source records",source.loc[source["in_study_years"]]),
                   ("Source records in study fires",shared),
                   ("Represented source records",source.loc[source["represented"]])]:
    engaged=frame.loc[frame["y"].notna()]
    for dimension in ["fireline_type","bin_width","bin_length","geometry_part_group"]:
        for weight in [None,"geodesic_length_m"]:
            t=binary_summary(engaged,[dimension],name,weight=weight)
            t=t.rename(columns={dimension:"group"});t.insert(0,"dimension",dimension)
            original_attribute_profiles.append(t)
tables["original_record_attribute_outcomes"]=pd.concat(original_attribute_profiles,ignore_index=True)

engaged_shared=shared.loc[shared["y"].notna()]
score_a,_=fire_scores(engaged_shared.loc[~engaged_shared["represented"]])
score_b,_=fire_scores(engaged_shared.loc[engaged_shared["represented"]])
row,detail=paired_score_difference(score_a["score"],score_b["score"],
                                  "Represented minus unrepresented engaged source records, same fires")
tables["selection_paired_fire_difference"]=pd.DataFrame([row])
tables["selection_paired_fire_details"]=detail

width_coverage=[]
for pop,mask in population_masks.items():
    part=source.loc[mask]
    for dim in ["Year","_engagement_norm","fireline_type"]:
        for value,stratum in part.groupby(dim,dropna=False):
            for status,group in stratum.groupby("width_status",dropna=False):
                width_coverage.append({"population":pop,"dimension":dim,"group":value,
                                       "width_status":status,"n_source_records":len(group),
                                       "stratum_source_records":len(stratum),
                                       "fraction":len(group)/len(stratum),
                                       "n_fires":group["_irwin_norm"].nunique()})
tables["original_width_availability"]=pd.DataFrame(width_coverage)
tables["source_numeric_attribute_audit"]=pd.DataFrame([
 {"quantity":col,**describe_numbers(source[col])}
 for col in ["LineWidthFeet","reported_length_m","stored_geodesic_length_m","source_length_5070_m","geodesic_length_m"]])
display(tables["population_inventory"])
display(tables["source_binary_outcomes_by_population"])


## 5. Retained portions, multiple prediction times, and source-record summaries

Each represented source record receives one row below. Source labels and construction attributes come directly from that source record. Environmental means use mapped-length weights over **retained fireline units with a valid value for that predictor**. Missing portions do not contribute to the mean; minimum, maximum, valid-length fraction, and historical time span remain available.

An environmental mean over several prediction times is a descriptive summary, not a forecast that existed at one date. Source records with one prediction time, and records with high retained-length ratios, are therefore examined separately.

The ratio of summed retained fireline-unit length to recomputed source length is diagnostic. Different length calculations and possible geometry overlap prevent calling it exact spatial coverage.


In [ ]:
require(units.groupby("source_record_key")["y"].nunique().eq(1).all(),"A source record has mixed inherited labels.")
require(units.groupby("_irwin_norm")["outer_fold"].nunique().eq(1).all(),"Saved folds split a fire.")
retained=units.groupby("source_record_key").agg(
 n_retained_units=("unit_uid","size"),retained_length_m=("actual_line_length_m","sum"),
 n_prediction_times=("prediction_anchor_utc","nunique"),
 first_retained_anchor=("prediction_anchor_utc","min"),
 last_retained_anchor=("prediction_anchor_utc","max"))
retained["anchor_span_hours"]=(retained["last_retained_anchor"]-retained["first_retained_anchor"]).dt.total_seconds()/3600
parents=source.loc[source["represented"]].merge(retained,on="source_record_key",validate="one_to_one")
parents["retained_length_ratio"]=parents["retained_length_m"]/parents["geodesic_length_m"]
parents["retention_ratio_review"]=parents["retained_length_ratio"].gt(RETENTION_RATIO_REVIEW_LIMIT)
parents["retention_ratio_group"]=numeric_bins(
 parents["retained_length_ratio"],[0,.25,.5,.8,.95,1.02,np.inf],
 ["<25%","25–<50%","50–<80%","80–<95%","95–102%","Above102%"],(0,None))
parents["time_span_group"]=np.select(
 [parents["n_prediction_times"].eq(1),parents["anchor_span_hours"].le(24),
  parents["anchor_span_hours"].le(48)],["One anchor","Multiple anchors, <=24 h","24–48 h"],default="More than48 h")
# Explicitly retain width/length/geometry diagnostics with each existing fireline unit.
units=units.merge(source[["source_record_key","geodesic_length_m","bin_length","bin_width",
                         "width_status","geographic_box","geometry_part_group"]],
                  on="source_record_key",how="left",validate="many_to_one")
units=units.merge(retained[["n_retained_units","retained_length_m"]],on="source_record_key",how="left",validate="many_to_one")
units["w_parent_equal"]=1.0/units["n_retained_units"]
units["w_parent_length_equal"]=units["actual_line_length_m"]/units["retained_length_m"]
require(np.allclose(units.groupby("source_record_key")["w_parent_equal"].sum(),1),"Parent-equal weights do not sum to one.")
require(np.allclose(units.groupby("source_record_key")["w_parent_length_equal"].sum(),1),"Parent-length allocation does not sum to one.")
for key,spec in PREDICTORS.items():
    col,label,unit,edges,labels,bounds=spec
    x=pd.to_numeric(units[col],errors="coerce")
    valid=np.isfinite(x)
    lo,hi=bounds
    if lo is not None: valid &= x.ge(lo)
    if hi is not None: valid &= x.le(hi)
    units["bin_"+key]=numeric_bins(units[col],edges,labels,bounds)
    work=pd.DataFrame({"source_record_key":units["source_record_key"],
                       "valid_length":units["actual_line_length_m"].where(valid,0.),
                       "weighted_value":(units["actual_line_length_m"]*x).where(valid,0.),
                       "value":x.where(valid)})
    agg=work.groupby("source_record_key").agg(
        valid_length=("valid_length","sum"),weighted_sum=("weighted_value","sum"),
        minimum=("value","min"),maximum=("value","max"))
    agg[key+"_retained_length_mean"]=agg["weighted_sum"]/agg["valid_length"].replace(0,np.nan)
    agg[key+"_valid_length_fraction"]=agg["valid_length"]/retained["retained_length_m"]
    agg=agg.rename(columns={"minimum":key+"_minimum","maximum":key+"_maximum"})
    cols=[key+"_retained_length_mean",key+"_minimum",key+"_maximum",key+"_valid_length_fraction"]
    parents=parents.merge(agg[cols],on="source_record_key",how="left",validate="one_to_one")
    parents["bin_"+key]=numeric_bins(parents[key+"_retained_length_mean"],edges,labels,bounds)

tables["represented_source_retention_summary"]=pd.DataFrame([
 {"quantity":col,**describe_numbers(parents[col])}
 for col in ["n_retained_units","retained_length_m","geodesic_length_m",
             "retained_length_ratio","n_prediction_times","anchor_span_hours"]])
tables["represented_source_retention_groups"]=binary_summary(parents,["retention_ratio_group"],"Retained-length ratio diagnostic")
tables["represented_source_timing_groups"]=binary_summary(parents,["time_span_group"],"Prediction-time spread diagnostic")
tables["parent_predictor_spread_summary"]=pd.DataFrame([
 {"predictor":key,**describe_numbers(parents[key+"_maximum"]-parents[key+"_minimum"])}
 for key in PREDICTORS])
# These dates document mapping/database activity, NOT verified fireline construction.
date_rows=[]
for col in ["LineDateTime","CreateDate","DateCurrent"]:
    original=pd.to_datetime(source[col],errors="coerce",utc=True)
    plausible=original.dt.year.between(1900,2100)
    parsed=pd.to_datetime(parents[col],errors="coerce",utc=True)
    parent_plausible=parsed.dt.year.between(1900,2100)
    date_rows.append({"date_field":col,"source_nonmissing":int(original.notna().sum()),
                     "source_missing_fraction":float(original.isna().mean()),
                     "source_outside_1900_2100":int((original.notna() & ~plausible).sum()),
                     "represented_plausible_dates":int(parent_plausible.sum()),
                     "dates_after_earliest_retained_anchor":int((parent_plausible & parsed.gt(parents["first_retained_anchor"])).sum()),
                     "dates_after_latest_retained_anchor":int((parent_plausible & parsed.gt(parents["last_retained_anchor"])).sum()),
                     "meaning":"Mapping/database date; not construction or proven historical availability"})
tables["mapping_date_availability"]=pd.DataFrame(date_rows)
display(tables["represented_source_retention_summary"])
display(tables["represented_source_timing_groups"])


## 6. Matched-data weighting comparisons

These schemes keep the existing source records and fireline units fixed:
- **Equal fireline units:** each child contributes one observation.
- **Equal total source-record weight:** each child receives 1 divided by its source record's retained child count.
- **Retained mapped length:** a child contributes its mapped length.
- **Equal source-record weight allocated by length:** each source record contributes a total of one, shared among its children according to their lengths.

Overall, equal total source-record weight must reproduce one row per represented source record, because every child inherits the same outcome. For condition groups, a source record can contribute to several groups. The weighting changes the question; it is not a correction that makes every alternative wrong.

Separate parent-summary profiles below change both the observation level and the environmental summary. They must not be confused with merely reweighting the same child values.


In [ ]:
WEIGHTINGS={
 "Equal fireline units":None,
 "Equal total source-record weight":"w_parent_equal",
 "Retained mapped length":"actual_line_length_m",
 "Equal source records, allocated by retained length":"w_parent_length_equal",
}
weighting_rows=[]; weighting_scores={}
for label,weight in WEIGHTINGS.items():
    t=binary_summary(units,[],label,weight)
    weighting_rows.append(t)
    weighting_scores[label]=fire_scores(units,weight)[0]["score"]
parent_result=binary_summary(parents,[],"One row per represented source record")
weighting_rows.append(parent_result)
tables["matched_population_weighting_comparison"]=pd.concat(weighting_rows,ignore_index=True)
parent_scores=fire_scores(parents)[0]["score"]
require(np.allclose(weighting_scores["Equal total source-record weight"].sort_index(),parent_scores.sort_index()),
        "Parent-weight allocation does not reproduce source-record outcome fractions.")

differences=[]; details=[]
for label in list(WEIGHTINGS)[1:]:
    row,detail=paired_score_difference(weighting_scores["Equal fireline units"],weighting_scores[label],
                                      label+" minus equal fireline units")
    differences.append(row);details.append(detail)
tables["weighting_paired_fire_differences"]=pd.DataFrame(differences)
tables["weighting_paired_fire_details"]=pd.concat(details,ignore_index=True)

weighted_profiles=[]
for key in list(PREDICTORS)+["width","length","fireline_type"]:
    group="fireline_type" if key=="fireline_type" else "bin_"+key
    for label,weight in WEIGHTINGS.items():
        t=binary_summary(units,[group],label,weight)
        t=t.rename(columns={group:"group"});t.insert(0,"predictor",key)
        weighted_profiles.append(t)
tables["condition_profiles_by_weighting"]=pd.concat(weighted_profiles,ignore_index=True)

parent_subsets={
 "All represented source records":pd.Series(True,index=parents.index),
 "One retained prediction time":parents["n_prediction_times"].eq(1),
 "High retained-length ratio":parents["retained_length_ratio"].between(HIGH_RETAINED_LENGTH_RATIO,RETENTION_RATIO_REVIEW_LIMIT),
 "One time and high retained-length ratio":parents["n_prediction_times"].eq(1)&parents["retained_length_ratio"].between(HIGH_RETAINED_LENGTH_RATIO,RETENTION_RATIO_REVIEW_LIMIT),
}
parent_profiles=[]; parent_subset_counts=[]
for name,mask in parent_subsets.items():
    part=parents.loc[mask]
    parent_subset_counts.append({"subset":name,"n_source_records":len(part),"n_fires":part["_irwin_norm"].nunique(),
                                "n_held":int(part["y"].sum()),"n_burned_over":int(len(part)-part["y"].sum())})
    for key in PREDICTORS:
        t=binary_summary(part,["bin_"+key],name)
        t=t.rename(columns={"bin_"+key:"group"});t.insert(0,"predictor",key)
        parent_profiles.append(t)
tables["retained_parent_summary_profiles"]=pd.concat(parent_profiles,ignore_index=True)
tables["retained_parent_subset_counts"]=pd.DataFrame(parent_subset_counts)

# Source-wide combinations concern construction attributes only: no invented weather for excluded records.
construction_combinations=[]
for name,frame in [("2020–2024 source records",source.loc[source["in_study_years"] & source["y"].notna()]),
                   ("Represented source records",parents)]:
    for group in ["bin_width","bin_length"]:
        t=binary_summary(frame,["fireline_type",group],name,intervals=False)
        t=t.rename(columns={group:"group"});t.insert(0,"attribute",group)
        construction_combinations.append(t)
tables["construction_attribute_combinations"]=pd.concat(construction_combinations,ignore_index=True)
environment_combinations=[]
for label,weight in WEIGHTINGS.items():
    t=binary_summary(units,["bin_humidity","bin_gust"],label,weight)
    environment_combinations.append(t)
tables["humidity_gust_combinations_by_weighting"]=pd.concat(environment_combinations,ignore_index=True)
display(tables["matched_population_weighting_comparison"])
display(tables["weighting_paired_fire_differences"])


## 7. Within-fire contrasts, year/type slices, and small-fragment sensitivity

The four 08A contrasts are repeated under the alternative weights, with an additional **line-sampled crown-depth** contrast. To guard against many children from a single source record masquerading as broad support, each side must have at least five retained fireline units **and two source records** in each compared fire. Consequently, counts may differ from 08A.

Each comparison uses the same eligible fires for its two sides and for all four weighting schemes. Year/type slices are descriptive, not independent validation. Their small groups are kept visible; do not select only favorable slices.

Original-record comparisons also examine reported width, mapped source-record length, and completed dozer versus hand firelines within the same fires. Each side must have at least five source records per fire. Equal-record and mapped-length comparisons use the same records and fires. The contrasts are repeated in the full 2020–2024 archive, the study fires, and the represented source records; they are descriptive associations, not construction prescriptions.

Minimum fireline-unit lengths of 5, 25, and 100 m are diagnostic exclusions, not new preferred preprocessing rules. The frozen population remains unchanged.


In [ ]:
CONTRASTS=[
 ("Humidity 60–100% versus <20%","humidity",["0–<20"],["60–<80","80–100"]),
 ("Gust >=10 versus <5 m/s","gust",["0–<5"],["10–<15","15–<20","20+"]),
 ("Opposite-side crown >=20 versus <10 m","opposite_crown",["0–<5","5–<10"],["20–<30","30+"]),
 ("Line crown >=20 versus <10 m","line_crown",["0–<5","5–<10"],["20–<30","30+"]),
 ("Reported width >=6 versus <3 m","width",[">0–<3"],["6–<12","12–<30","30+"]),
]
def condition_contrast(frame,spec,weight,label):
    name,key,low,high=spec
    a=frame["bin_"+key].isin(low);b=frame["bin_"+key].isin(high)
    require(not(a&b).any(),"Overlapping contrast groups")
    eligible_sides=[]
    for mask in [a,b]:
        counts=frame.loc[mask].groupby("_irwin_norm").agg(
            n_units=("unit_uid","size"),n_parents=("source_record_key","nunique"))
        eligible_sides.append(set(counts.index[
            counts["n_units"].ge(MIN_CONTRAST_UNITS_PER_SIDE)&
            counts["n_parents"].ge(MIN_CONTRAST_PARENTS_PER_SIDE)]))
    fires=eligible_sides[0]&eligible_sides[1]
    left=frame.loc[a & frame["_irwin_norm"].isin(fires)]
    right=frame.loc[b & frame["_irwin_norm"].isin(fires)]
    sa=fire_scores(left,weight)[0]["score"]
    sb=fire_scores(right,weight)[0]["score"]
    row,detail=paired_score_difference(sa,sb,name+" | "+label)
    row.update(contrast=name,weighting=label,n_units_A=len(left),n_units_B=len(right),
               n_source_records_A=left["source_record_key"].nunique(),
               n_source_records_B=right["source_record_key"].nunique(),
               meets_fire_screen=len(fires)>=MIN_GROUP_FIRES)
    detail["contrast"]=name;detail["weighting"]=label
    return row,detail

contrast_rows=[]; contrast_details=[]
for spec in CONTRASTS:
    for label,weight in WEIGHTINGS.items():
        row,detail=condition_contrast(units,spec,weight,label)
        contrast_rows.append(row);contrast_details.append(detail)
tables["within_fire_contrasts_by_weighting"]=pd.DataFrame(contrast_rows)
tables["within_fire_contrast_details"]=pd.concat(contrast_details,ignore_index=True)
slice_rows=[]
for dimension in ["Year","fireline_type"]:
    for value,part in units.groupby(dimension):
        for spec in CONTRASTS[:2]:
            row,_=condition_contrast(part,spec,"w_parent_equal","Equal total source-record weight")
            row.update(slice_dimension=dimension,slice_group=value)
            slice_rows.append(row)
tables["within_fire_weather_contrasts_by_year_and_type"]=pd.DataFrame(slice_rows)

fragment_rows=[]
for threshold in [0,5,25,100]:
    part=units.loc[units["actual_line_length_m"].ge(threshold)].copy()
    # Recompute parent totals on the retained diagnostic subset.
    part["w_parent_equal"]=1/part.groupby("source_record_key")["unit_uid"].transform("size")
    for label,weight in [("Equal fireline units",None),("Equal source records in subset","w_parent_equal"),
                         ("Retained mapped length","actual_line_length_m")]:
        result=binary_summary(part,[],label,weight)
        result.insert(0,"minimum_unit_length_m",threshold)
        fragment_rows.append(result)
tables["minimum_unit_length_sensitivity"]=pd.concat(fragment_rows,ignore_index=True)
fragment_contrasts=[]
for threshold in [0,5,25,100]:
    part=units.loc[units["actual_line_length_m"].ge(threshold)].copy()
    part["w_parent_equal"]=1/part.groupby("source_record_key")["unit_uid"].transform("size")
    for spec in CONTRASTS[:2]:
        row,_=condition_contrast(part,spec,"w_parent_equal","Equal source records in subset")
        row["minimum_unit_length_m"]=threshold;fragment_contrasts.append(row)
tables["weather_contrast_fragment_sensitivity"]=pd.DataFrame(fragment_contrasts)

# Original-record contrasts do not inherit the number of 250 m children.
# Both weighting schemes use the same valid-length records and eligible fires.
ORIGINAL_ATTRIBUTE_CONTRASTS=[
 ("Reported width >=6 versus <3 m","bin_width",[">0–<3"],["6–<12","12–<30","30+"]),
 ("Mapped source-record length >=1000 versus <50 m","bin_length",[">0–<50"],["1000+"]),
 ("Completed Dozer Line versus Completed Hand Line","fireline_type",["Completed Hand Line"],["Completed Dozer Line"]),
]
original_contrast_rows=[]; original_contrast_details=[]
for population,mask in list(population_masks.items())[1:]:
    frame=source.loc[mask & source["y"].notna()].copy()
    valid_length=np.isfinite(frame["geodesic_length_m"]) & frame["geodesic_length_m"].gt(0)
    excluded_length_records=int((~valid_length).sum())
    frame=frame.loc[valid_length]
    for name,column,group_a,group_b in ORIGINAL_ATTRIBUTE_CONTRASTS:
        a=frame[column].isin(group_a);b=frame[column].isin(group_b)
        require(not(a&b).any(),"Overlapping original-record contrast groups")
        support_a=frame.loc[a].groupby("_irwin_norm").size()
        support_b=frame.loc[b].groupby("_irwin_norm").size()
        fires=set(support_a.index[support_a.ge(MIN_ORIGINAL_RECORDS_PER_SIDE)]) & set(
            support_b.index[support_b.ge(MIN_ORIGINAL_RECORDS_PER_SIDE)])
        left=frame.loc[a & frame["_irwin_norm"].isin(fires)]
        right=frame.loc[b & frame["_irwin_norm"].isin(fires)]
        for weighting,weight in [("Equal source records",None),("Mapped source length","geodesic_length_m")]:
            scores_a=fire_scores(left,weight)[0]["score"]
            scores_b=fire_scores(right,weight)[0]["score"]
            row,detail=paired_score_difference(scores_a,scores_b,population+" | "+name+" | "+weighting)
            row.update(population=population,contrast=name,weighting=weighting,
                       group_A="; ".join(group_a),group_B="; ".join(group_b),
                       n_source_records_A=len(left),n_source_records_B=len(right),
                       population_records_excluded_invalid_length=excluded_length_records,
                       meets_fire_screen=len(fires)>=MIN_GROUP_FIRES)
            detail["population"]=population;detail["contrast"]=name;detail["weighting"]=weighting
            original_contrast_rows.append(row);original_contrast_details.append(detail)
tables["original_record_within_fire_contrasts"]=pd.DataFrame(original_contrast_rows)
tables["original_record_within_fire_contrast_details"]=pd.concat(original_contrast_details,ignore_index=True)
display(tables["original_record_within_fire_contrasts"])

tables["unit_length_distribution"]=pd.DataFrame([describe_numbers(units["actual_line_length_m"])])
display(tables["within_fire_contrasts_by_weighting"])
display(tables["minimum_unit_length_sensitivity"])


## 8. Canopy availability and mapping-date caveats

Opposite-side canopy summaries require a reliable incoming-fire direction. Missing values must not be interpreted as bare ground or zero canopy.

We compare canopy sampled **on** the fireline with canopy sampled on the **opposite side**, and repeat the profiles on the same direction-valid population. The locations and summary statistics differ, so the two predictors are not interchangeable measurements. This is an availability/interpretation check, not a model-importance experiment.

No missing widths or canopy values are imputed. Mapping dates and source attributes are not automatically assumed to have been available before the historical prediction time.


In [ ]:
opposite_col=PREDICTORS["opposite_crown"][0]
line_col=PREDICTORS["line_crown"][0]
direction_valid=units["landfire_directional_context_valid"].fillna(False).astype(bool)
units["canopy_availability"]=np.select(
 [units[opposite_col].notna() & units[line_col].notna(),
  units[opposite_col].isna() & units[line_col].notna()],
 ["Both locations available","Line available; opposite-side missing"],default="Other missingness")
distance_small=units[PREDICTORS["distance"][0]].lt(25)
canopy_checks={
 "opposite_missing":int(units[opposite_col].isna().sum()),
 "line_crown_missing":int(units[line_col].isna().sum()),
 "direction_invalid":int((~direction_valid).sum()),
 "opposite_missing_and_distance_below25":int((units[opposite_col].isna() & distance_small).sum()),
 "opposite_missing_equals_distance_below25":bool(units[opposite_col].isna().equals(distance_small)),
}
tables["canopy_availability_checks"]=pd.DataFrame([canopy_checks])
tables["canopy_availability_outcomes"]=binary_summary(units,["canopy_availability"],"Canopy availability diagnostic")
canopy_profiles=[]
for population,mask in [
 ("All retained fireline units",pd.Series(True,index=units.index)),
 ("Common direction-valid and both-canopy population",
  direction_valid & units[opposite_col].notna() & units[line_col].notna())]:
    for key in ["opposite_crown","line_crown","line_cover"]:
        t=binary_summary(units.loc[mask],["bin_"+key],population,"w_parent_equal")
        t=t.rename(columns={"bin_"+key:"group"});t.insert(0,"predictor",key)
        canopy_profiles.append(t)
tables["canopy_profiles_common_population"]=pd.concat(canopy_profiles,ignore_index=True)
display(tables["canopy_availability_checks"])
print("Missing opposite-side canopy is a measurement-context issue, not evidence of absent vegetation.")


## 9. Compare the results visually

Fraction plots use fixed 0–100% scales; contrast plots show differences in percentage points. Whole-fire uncertainty is shown where available. The source-population figure shows record fractions, not attrition through comparable rows. Weather-contrast panels compare identical eligible fires across weighting schemes.

The full tables remain the authoritative source for counts, missingness, and sparse-group flags.


In [ ]:
def save_figure(fig,name):
    if IS_SMOKE_TEST:
        fig.text(.99,.005,f"SMOKE TEST: {BOOTSTRAP_REPLICATES} resamples",
                 ha="right",va="bottom",fontsize=8,color="#A04000",
                 bbox={"facecolor":"white","edgecolor":"none"})
    for ext in ["png","pdf"]:
        path=FIG_DIR/(name+"."+ext)
        fig.savefig(path,bbox_inches="tight",facecolor="white")
        figures.append(path)
    plt.show();plt.close(fig)

composition=tables["source_three_outcome_composition"].pivot(
 index="population",columns="outcome",values="record_fraction").reindex(list(population_masks))
fig,ax=plt.subplots(figsize=(10,4.8))
left=np.zeros(len(composition))
for outcome,color in zip(OUTCOMES,["#0072B2","#D55E00","#999999"]):
    values=composition[outcome].to_numpy()
    ax.barh(np.arange(len(values)),values,left=left,label=outcome,color=color,height=.65)
    for i,value in enumerate(values):
        if value>.045: ax.text(left[i]+value/2,i,f"{value:.1%}",ha="center",va="center",color="white",fontsize=9)
    left+=values
ax.set(yticks=np.arange(len(composition)),yticklabels=composition.index,xlim=(0,1),
       xlabel="Fraction of source records (three recorded outcomes)",
       title="Original source records and represented source records")
ax.invert_yaxis();ax.xaxis.set_major_formatter(PercentFormatter(1))
ax.legend(loc="upper center",bbox_to_anchor=(.5,-.16),ncol=3)
save_figure(fig,"08B_source_population_outcomes")

weights=tables["matched_population_weighting_comparison"].iloc[:4]
short=["Equal fireline units","Equal source records","Retained mapped length","Equal records, length allocation"]
fig,ax=plt.subplots(figsize=(10,4.5))
for i,row in weights.reset_index(drop=True).iterrows():
    ax.plot(row["pooled_held_rate"],i-.12,"s",color="#D55E00",label="Pooled" if i==0 else None)
    ax.plot(row["fire_average_held_rate"],i+.12,"o",color="#0072B2",label="Fire-average" if i==0 else None)
    ax.hlines(i+.12,row["held_ci_lower"],row["held_ci_upper"],color="#0072B2",linewidth=2)
ax.set(yticks=np.arange(4),yticklabels=short,xlim=(0,1),
       xlabel="Recorded Held fraction (Held and Burned Over only)",
       title="Same retained data, different weighting questions")
ax.xaxis.set_major_formatter(PercentFormatter(1));ax.invert_yaxis()
ax.legend(loc="lower left")
save_figure(fig,"08B_matched_weighting_comparison")

fig,axes=plt.subplots(1,2,figsize=(13,4.8))
profile=tables["condition_profiles_by_weighting"]
for ax,key,title in zip(axes,["humidity","gust"],
                        ["Mean forecast humidity, 0–18 h (%)","Forecast gust at +18 h (m/s)"]):
    labels=PREDICTORS[key][4]
    for offset,(label,color,marker) in enumerate([
        ("Equal fireline units","#0072B2","o"),
        ("Equal total source-record weight","#D55E00","s"),
        ("Retained mapped length","#009E73","^")]):
        part=profile.loc[profile["predictor"].eq(key)&profile["analysis"].eq(label)].set_index("group").reindex(labels)
        for j,(_,row) in enumerate(part.iterrows()):
            if pd.isna(row["fire_average_held_rate"]): continue
            x=j+(offset-1)*.16
            use_color=color if bool(row["meets_reporting_screen"]) else "#888888"
            ax.plot(x,row["fire_average_held_rate"],marker=marker,color=use_color,linestyle="none")
            if np.isfinite(row["held_ci_lower"]) and np.isfinite(row["held_ci_upper"]):
                ax.vlines(x,row["held_ci_lower"],row["held_ci_upper"],color=use_color,linewidth=1)
        ax.plot([],[],marker=marker,color=color,linestyle="none",label=short[offset])
    ax.set(xticks=range(len(labels)),xticklabels=labels,ylim=(0,1),xlabel=title)
    ax.yaxis.set_major_formatter(PercentFormatter(1));ax.tick_params(axis="x",labelrotation=30)
axes[0].set_ylabel("Fire-average recorded Held fraction")
axes[1].legend(loc="lower right",fontsize=9)
fig.suptitle("Condition profiles under three weighting schemes")
fig.tight_layout()
save_figure(fig,"08B_weather_weighting_profiles")

contrasts=tables["within_fire_contrasts_by_weighting"]
fig,axes=plt.subplots(1,2,figsize=(13,4.7),sharex=True)
for ax,contrast in zip(axes,[CONTRASTS[0][0],CONTRASTS[1][0]]):
    part=contrasts.loc[contrasts["contrast"].eq(contrast)].set_index("weighting").reindex(list(WEIGHTINGS))
    for i,(_,row) in enumerate(part.iterrows()):
        if not np.isfinite(row["difference_B_minus_A"]): continue
        color="#0072B2" if row["meets_fire_screen"] else "#888888"
        ax.plot(100*row["difference_B_minus_A"],i,"o",color=color)
        if np.isfinite(row["ci_lower"]) and np.isfinite(row["ci_upper"]):
            ax.hlines(i,100*row["ci_lower"],100*row["ci_upper"],color=color,linewidth=2)
    ax.axvline(0,color="#555555",linestyle="--",linewidth=1)
    ax.set(yticks=range(4),yticklabels=short,title=contrast,
           xlabel="Difference in Held fraction (percentage points)")
    ax.invert_yaxis()
fig.tight_layout()
save_figure(fig,"08B_weather_contrast_weighting")


## 10. Save the run, checks, and research roadmap

The exported source table contains original-record attributes and audit flags. The parent-summary table describes retained portions; it is **not a replacement prediction dataset**. The unit table preserves the existing prediction anchors and adds diagnostic weights.

The report names remaining research tracks explicitly. It does not claim to have tested every combination, reconstructed complete constructed firelines, or validated a new decision rule.


In [ ]:
require(len(units)==304143 and units["source_record_key"].nunique()==142999,"Final unit population changed.")
require(digest(units,unit_base_columns,["unit_uid"])==units_start_hash,"Original unit data or prediction anchors changed.")
require(digest(source,source_columns,["_irwin_norm","_geom_hash"])==source_start_hash,"Original source attributes changed.")
require(len(parents)==142999 and parents["source_record_key"].is_unique,"Parent summary changed source membership.")
for p in input_paths:
    require(p.stat().st_size==input_stats[str(p)]["bytes"] and
            p.stat().st_mtime_ns==input_stats[str(p)]["modified_ns"],"Input changed during run: "+str(p))
for name,table in tables.items():
    table.to_csv(RUN_DIR/(name+".csv"),index=False,encoding="utf-8-sig")
source.to_parquet(RUN_DIR/"08B_original_source_record_audit.parquet",index=False)
parents.to_parquet(RUN_DIR/"08B_retained_source_record_summaries.parquet",index=False)
units.to_parquet(RUN_DIR/"08B_existing_units_with_diagnostic_weights.parquet",index=False)
notebook_source_hash=None
if NOTEBOOK_PATH.is_file():
    document=json.loads(NOTEBOOK_PATH.read_text(encoding="utf-8"))
    notebook_source_hash=hashlib.sha256(json.dumps([x["source"] for x in document["cells"]],ensure_ascii=False).encode()).hexdigest()
config={
 "build_id":BUILD_ID,"run_id":RUN_ID,"completed_utc":datetime.now(timezone.utc).isoformat(),
 "is_smoke_test":IS_SMOKE_TEST,"input_08A_run":INPUT_08A_RUN_NAME,"input_files":input_stats,
 "python":sys.version,"platform":platform.platform(),"bootstrap_replicates":BOOTSTRAP_REPLICATES,
 "package_versions":{name:package_version(name) for name in required},
 "random_seed":RANDOM_SEED,"geometry_batch_size":GEOMETRY_BATCH_SIZE,"geometry_reference":"WGS84 ellipsoid",
 "source_years":SOURCE_YEARS,"comparison_years":COMPARISON_YEARS,
 "width_review_ft":WIDTH_REVIEW_FT,"high_retained_length_ratio":HIGH_RETAINED_LENGTH_RATIO,
 "retention_ratio_review_limit":RETENTION_RATIO_REVIEW_LIMIT,
 "minimum_group_rows":MIN_GROUP_ROWS,"minimum_group_fires":MIN_GROUP_FIRES,
 "minimum_group_outcome_count":MIN_GROUP_OUTCOME_COUNT,
 "minimum_contrast_units_per_side":MIN_CONTRAST_UNITS_PER_SIDE,
 "minimum_contrast_parents_per_side":MIN_CONTRAST_PARENTS_PER_SIDE,
 "minimum_original_records_per_side":MIN_ORIGINAL_RECORDS_PER_SIDE,
 "original_attribute_contrasts":ORIGINAL_ATTRIBUTE_CONTRASTS,
 "predictors":PREDICTORS,"width_edges":WIDTH_EDGES,"width_labels":WIDTH_LABELS,
 "length_edges":LENGTH_EDGES,"length_labels":LENGTH_LABELS,"contrasts":CONTRASTS,
 "weightings":WEIGHTINGS,"source_values_hash":source_start_hash,"unit_values_hash":units_start_hash,
 "notebook_source_hash":notebook_source_hash,
 "scope":"Original mapped records, weighting and selection sensitivity; no construction-line reconstruction or model training"}
write_json("08B_run_manifest.json",config)
write_json("08B_quality_checks.json",{
 "source_records_preserved":True,"source_attributes_preserved":True,
 "frozen_units_and_prediction_times_preserved":True,"all_source_joins_match":True,
 "parent_weights_sum_to_one":True,"parent_weighting_reproduces_parent_outcomes":True,
 "not_engaged_kept_separate":True,"original_input_stats_unchanged":True,
 "geometry_records_audited":len(geometry),"geometry_issue_records":int(source["geometry_issue"].ne("ok").sum()),
 "retention_ratios_above_review_limit":int(parents["retention_ratio_review"].sum()),
 "is_smoke_test":IS_SMOKE_TEST})
def mdrow(values):
    return "| "+" | ".join(str(v).replace("|","/").replace("\n"," ") for v in values)+" |"
lines=[
 "# 08B — Original mapped firelines, scale, and selection sensitivity","",
 "**SMOKE TEST — not a paper result.**" if IS_SMOKE_TEST else "**Exploratory audit; no models fitted.**","",
 f"Run {RUN_ID}; {BOOTSTRAP_REPLICATES:,} whole-fire bootstrap replicates.",
 "The source archive contains outcome-classified mapped records, not verified complete constructed firelines.",
 "No geometry gaps were connected, no records were merged into inferred assets, and no outcomes were changed.","",
 "## Population inventory",
 mdrow(["Population","Observation level","Count","Fires"]),"|---|---|---:|---:|"]
for r in tables["population_inventory"].itertuples():
    lines.append(mdrow([r.population,r.observation_level,f"{r.n_observations:,}",r.n_fires]))
lines += ["","Not Engaged is a separate original outcome. Binary rates always exclude it.","",
          "## Does the counting basis matter?",
          mdrow(["Counting basis","Pooled Held","Fire-average Held"]),"|---|---:|---:|"]
for r in tables["matched_population_weighting_comparison"].itertuples():
    lines.append(mdrow([r.analysis,f"{r.pooled_held_rate:.1%}",f"{r.fire_average_held_rate:.1%}"]))
lines += ["",
 "These are different estimands on the same retained records, not competing claims about one universal success rate.",
 "Paired differences and intervals are saved in weighting_paired_fire_differences.csv.","",
 "## What an original-record summary represents",
 f"- {int(parents['n_prediction_times'].gt(1).sum()):,} represented source records contain more than one retained prediction time.",
 f"- {int(parents['anchor_span_hours'].gt(24).sum()):,} span more than 24 hours.",
 f"- {int(parents['retention_ratio_review'].sum()):,} retained-length ratios exceed the {RETENTION_RATIO_REVIEW_LIMIT:g} review limit.",
 f"- {int(units['actual_line_length_m'].lt(5).sum()):,} retained fireline units are shorter than 5 m.",
 "Parent environmental means summarize valid retained portions at their actual historical times; missing portions are excluded and the means are not a new common-time forecast.",
 "Retained-length ratio is a sum-of-lengths diagnostic, not verified spatial coverage.","",
 "## Weather contrasts under alternative weighting",
 mdrow(["Contrast","Weighting","Fires","Difference (percentage points)","95% interval"]),"|---|---|---:|---:|---:|"]
for r in tables["within_fire_contrasts_by_weighting"].itertuples():
    if r.contrast in [CONTRASTS[0][0],CONTRASTS[1][0]]:
        interval=f"{100*r.ci_lower:.1f} to {100*r.ci_upper:.1f}" if np.isfinite(r.ci_lower) else "Unavailable"
        lines.append(mdrow([r.contrast,r.weighting,r.n_common_fires,f"{100*r.difference_B_minus_A:.1f}",interval]))
lines += ["",
 "Every side requires at least five fireline units and two source records per fire. Counts may therefore differ from 08A.",
 "Compare directions and magnitudes across schemes, years and types; do not retain only intervals excluding zero.","",
 "## Construction attributes and geometry",
 "original_record_attribute_outcomes.csv gives width, recomputed mapped length, type, and stored-part profiles in the broader archive and selected records.",
 "original_width_availability.csv exposes missingness by population, year, outcome and type.",
 "geometry_length_diagnostics.csv compares stored and recomputed lengths by coarse geographic box.",
 "original_record_within_fire_contrasts.csv compares reported width, mapped length, and fireline type within the same fires, using original records rather than child units.",
 "Each original-record contrast requires at least five records on each side in a fire. The bin choices are exploratory, not optimal construction specifications.",
 "Source-record length can itself reflect outcome-overlay splitting and does not establish optimal constructed-fireline length.",
 "Mapping dates are not construction dates or proof of when reported attributes became operationally available.","",
 "## Canopy availability",
 f"Opposite-side missing: {canopy_checks['opposite_missing']:,}; line-sampled crown missing: {canopy_checks['line_crown_missing']:,}.",
 f"Opposite-side missing matches distance below 25 m: {canopy_checks['opposite_missing_equals_distance_below25']}.",
 "Common-population canopy summaries compare different sampling locations; missing directional canopy is not absent vegetation.","",
 "## Research roadmap",
 "1. 08C: develop interpretable condition rules in development fires; freeze them and evaluate on excluded fires and forward years. Select observation level only after reviewing 08B. The previously explored archive remains exploratory; seek new fires for confirmation.",
 "2. Geometry-scale experiment: rebuild actual 100/250/500 m units, eligibility, timing, and predictors on a matched domain. Reweighting in 08B is not that experiment.",
 "3. Label-agreement study: derive NIROPS outcomes independently, without NIFC-driven compatibility selection, and compare before filtering disagreements.",
 "4. Construction-data study: seek verified width, construction times and stable asset identifiers; review original geometry revisions/overlaps before claiming complete constructed firelines or unique network kilometres.",
 "5. Broader environmental population: extract location- and time-appropriate predictors for unselected source records only after defining defensible timing. Do not assign them retained-child averages.",
 "6. Prospective study: choose candidates before outcomes are known and verify historical/live predictor availability.","",
 "No current result establishes a causal effect of building or widening a fireline, or a safe operational decision threshold."]
(RUN_DIR/"08B_FINDINGS_REPORT.md").write_text("\n".join(lines)+"\n",encoding="utf-8")
write_json("08B_output_inventory.json",[
 {"path":str(p.relative_to(RUN_DIR)),"bytes":p.stat().st_size} for p in sorted(RUN_DIR.rglob("*")) if p.is_file()])
display(Markdown("\n".join(lines)))
print("\nSaved run:",RUN_DIR)
print("Share 08B_FINDINGS_REPORT.md and the weighting/selection/retention tables for the next review.")


## After the run

Start with the source-population, matched-weighting, retained-timing, and paired-weather tables. Then examine width/type/length and canopy diagnostics.

An original-record analysis is valuable even when it changes the apparent association. The aim is to learn which conclusions persist under reasonable measurement choices and which require additional data, not to find a construction rule at any cost.

Keep 08A and the prior 05/06 model results unchanged. Decide the observation level and predictor definitions for **08C** after reviewing the saved 08B results.
